# PII Masking with NLP

In [79]:
!pip --quiet install langdetect 

In [80]:
import csv
from langdetect import detect
import pickle

In [81]:
from utils import *

### View Data

#### Note: we had a badline our in the dataset
Bad lines: 1
Bad lines occur at row numbers: [42759]

In [82]:
bad_row_numbers = []
with open('PII43k.csv', 'r') as f:
	reader = csv.reader(f)
	header = next(reader)
	expected_len = len(header)
	for idx, row in enumerate(reader, start=2):  # start=2 accounts for the header line
		if len(row) != expected_len:
			bad_row_numbers.append(idx)

print("Bad lines occur at row numbers:", bad_row_numbers)

Bad lines occur at row numbers: [42759]


In [83]:
df_full = pd.read_csv('PII43k.csv', on_bad_lines='skip')

In [84]:
print(df_full["Template"][0])
print(df_full["Filled Template"][0])

In our video conference, discuss the role of evidence in the arbitration process involving [FULLNAME_1] and [FULLNAME_2].
In our video conference, discuss the role of evidence in the arbitration process involving Dr. Marvin Rolfson and Julius Daugherty.


In [85]:


cleaned_matches, unique_matches = get_template_tokens(df_full)

print(unique_matches)
print(len(unique_matches))

print(cleaned_matches)
print(len(cleaned_matches))


{'CURRENCYSYMBOL_N', 'FIRSTNAME_6', 'CURRENCYCODE_1', 'LASTNAME_1', 'GENDER_1', 'EMAIL_N', 'IPV4_N', 'EMAIL_3', 'BITCOINADDRESS_1', 'IP_2', 'ACCOUNTNUMBER_N', 'STATE_1', 'AMOUNT_7', 'CITY_1', 'NUMBER_N', 'EMAIL_2', 'ACCOUNTNAME_N', 'FULLNAME_6', 'STREET_1', 'MAC_1', 'IPV4_2', 'FIRSTNAME_4', 'STATE_4', 'JOBTYPE_1', 'ACCOUNTNUMBER_1', 'ACCOUNTNAME_1', 'JOBTITLE_2', 'IP_1', 'CURRENCY_1', 'MAC_N', 'URL_4', 'NAME_3', 'COUNTY_1', 'SECONDARYADDRESS_1', 'CREDITCARDCVV_N', 'PASSWORD_2', 'PIN_1', 'ORDINALDIRECTION_2', 'LASTNAME_9', 'CURRENCYNAME_2', 'FIRSTNAME_3', 'NAME_N', 'STREET_2', 'BIC_N', 'USERNAME_3', 'NUMBER_4', 'STREETADDRESS_2', 'SECONDARYADDRESS_N', 'LITECOINADDRESS_N', 'FIRSTNAME_7', 'GENDER_2', 'NUMBER_3', 'JOBAREA_1', 'EMAIL_1', 'FIRSTNAME_9', 'FULLNAME_5', 'ACCOUNTNUMBER_2', 'AMOUNT_6', 'URL_5', 'STATE_3', 'LASTNAME_8', 'CREDITCARDNUMBER_2', 'IPV6_N', 'CURRENCYCODE_N', 'JOBAREA_3', 'FULLNAME_4', 'PASSWORD_1', 'NEARBYGPSCOORDINATE_1', 'PASSWORD_N', 'CURRENCY_3', 'AMOUNT_9', 'NUMBER

In [86]:

cleaned_name_tokens, name_tokens = get_token_tokens(df_full)
print(name_tokens)
print(len(name_tokens))
print(cleaned_name_tokens)
print(len(cleaned_name_tokens))

{'B-STATE', 'B-BUILDINGNUMBER', 'I-CURRENCY', 'B-NUMBER', 'I-PASSWORD', 'B-CREDITCARDCVV', 'B-FULLNAME', 'B-IBAN', 'B-IPV6', 'B-LASTNAME', 'I-ACCOUNTNUMBER', 'I-ZIPCODE', 'I-ACCOUNTNAME', 'B-IP', 'B-CITY', 'B-CURRENCYNAME', 'I-JOBTITLE', 'I-NEARBYGPSCOORDINATE', 'B-CURRENCY', 'I-CREDITCARDCVV', 'B-LITECOINADDRESS', 'I-CREDITCARDNUMBER', 'I-NAME', 'B-GENDER', 'I-JOBAREA', 'B-ZIPCODE', 'I-CURRENCYNAME', 'I-FIRSTNAME', 'B-MASKEDNUMBER', 'B-EMAIL', 'B-COUNTY', 'I-LITECOINADDRESS', 'B-BITCOINADDRESS', 'B-JOBAREA', 'I-LASTNAME', 'B-ACCOUNTNAME', 'B-CREDITCARDNUMBER', 'I-NUMBER', 'I-USERNAME', 'B-DISPLAYNAME', 'I-BUILDINGNUMBER', 'I-MAC', 'B-IPV4', 'B-NAME', 'I-ETHEREUMADDRESS', 'I-PIN', 'B-CURRENCYCODE', 'I-STREETADDRESS', 'B-FIRSTNAME', 'I-STREET', 'I-IPV4', 'I-BITCOINADDRESS', 'I-GENDER', 'B-MAC', 'B-JOBTYPE', 'B-USERAGENT', 'B-ETHEREUMADDRESS', 'B-JOBDESCRIPTOR', 'B-SEXTYPE', 'I-SECONDARYADDRESS', 'I-MASKEDNUMBER', 'B-ORDINALDIRECTION', 'B-PIN', 'I-IP', 'B-AMOUNT', 'B-URL', 'I-AMOUNT', 'I

In [87]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in cleaned_matches:
    pattern = r'\[' + token + r'_\d+\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

# Count occurrences of each cleaned token in the "Tokens" column of df_full
token_counts = {}
for token in cleaned_name_tokens:
    # Each row in "Tokens" is a list, so count token appearances per row.
    token_counts[token] = int(df_full["Tokens"].dropna().apply(lambda lst: lst.count(token)).sum())

# Convert counts to a DataFrame and sort for display
token_counts_df = pd.DataFrame(list(token_counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
token_counts_df

counts_df_renamed = counts_df.rename(columns={'Token':'template_entity', 'Count':'template_count'})
token_counts_df_renamed = token_counts_df.rename(columns={'Token':'token_entity', 'Count':'token_count'})

merged_df = pd.concat([counts_df_renamed.reset_index(drop=True),
                       token_counts_df_renamed.reset_index(drop=True)], axis=1)
merged_df

,template_entity,template_count,token_entity,token_count
0,FULLNAME,20195,NAME,157983
1,NAME,12494,FULLNAME,78058
2,EMAIL,7868,EMAIL,74760
3,CITY,7309,CITY,20870
4,JOBAREA,2743,URL,10928
5,FIRSTNAME,2525,NUMBER,10824
6,STATE,2285,IP,9677
7,STREETADDRESS,1051,STREET,6575
8,URL,982,USERAGENT,5763
9,USERNAME,553,STREETADDRESS,5500


In [90]:

df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME_1] to send ...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME_1] who wants...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [101]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[FULLNAME_1\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(10, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)



In our video conference, discuss the role of evidence in the arbitration process involving [FULLNAME_1] and [FULLNAME_2].
In our video conference, discuss the role of evidence in the arbitration process involving Dr. Marvin Rolfson and Julius Daugherty.


Discuss the options for [FULLNAME_1] who wants to obtain a green card through employment in [CITY_1].
Discuss the options for Jeffery Pfeffer who wants to obtain a green card through employment in Port Ron.


13. Write a press release announcing [FULLNAME_1]'s new mindfulness-based therapy practice in [CITY_1].
13. Write a press release announcing Gayle Waters's new mindfulness-based therapy practice in Oceanside.


9. Develop an inventory management plan for [FULLNAME_1] that ensures stock levels are maintained across multiple [CITY_N] locations.
9. Develop an inventory management plan for Evan Anderson that ensures stock levels are maintained across multiple Mayaguez locations.


During the video conference with [FULLNAME_1], discu

In [12]:
cleaned_matches, unique_matches = get_template_tokens(df_full)

def replace_unique_tokens(text, tokens):
	for token in tokens:
		# Match either an underscore with one or more digits or with 'N'
		pattern = r'\[' + token + r'_(?:\d+|N)\]'
		# Replace with the token in square brackets (e.g., "[NAME]")
		text = re.sub(pattern, f'[{token}]', text)
	return text

df_full['Template'] = df_full['Template'].apply(lambda t: replace_unique_tokens(t, cleaned_matches))

# view the first 5 rows of the 'Template' column
df_full['Template'].head()

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [FULLNAME] who wants t...
3    13. Write a press release announcing [FULLNAME...
4    9. Develop an inventory management plan for [F...
Name: Template, dtype: object

In [59]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[SEXTYPE\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(10, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


No rows found with the token [SEXTYPE]


In [60]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[SEX\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [61]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[GENDER\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(20, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)



Prepare a grant proposal for [NAME] [NAME] to study the effects of peer pressure on [GENDER] risk-taking behavior.
Prepare a grant proposal for Zola Doyle to study the effects of peer pressure on Neither risk-taking behavior.


Could you help me create a handout for [NAME] on supporting clients through [GENDER] transition?
Could you help me create a handout for Eloise O'Hara on supporting clients through Intersex man transition?


Discuss the impact of cultural differences on the perception of [GENDER]'s emotional expression during our video conference.
Discuss the impact of cultural differences on the perception of Trigender's emotional expression during our video conference.


8. Explain the role of forensic psychologists in the treatment and rehabilitation of [GENDER] offenders.
8. Explain the role of forensic psychologists in the treatment and rehabilitation of Hermaphrodite offenders.


In our video conference, we will discuss the latest research on the psychological effects of c

In [62]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[BUILDINGNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [63]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBAREA\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [64]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[DISPLAYNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [65]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ACCOUNTNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [66]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IP\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


7. HIPAA guidelines for protecting patients' [IP] in telemedicine consultations.
7. HIPAA guidelines for protecting patients' 146.229.205.216 in telemedicine consultations.


Hey, can you provide an overview of the privacy laws that apply to collecting [IP] and [IP] addresses?
Hey, can you provide an overview of the privacy laws that apply to collecting 45.41.218.105 and 2d7e:c6e8:db2e:50ae:c3fb:9623:af1f:9d69 addresses?


Hey, can you provide some tips on how to ensure our [URL] and [IP] infrastructure remains operational during a crisis?
Hey, can you provide some tips on how to ensure our https://definite-victim.net and 9cf2:ceaa:3bad:9e1a:e73c:cf34:336a:ae2f infrastructure remains operational during a crisis?


Please draft a privacy policy for [NAME]'s website that covers user data protection, including [USERNAME], [PASSWORD], and [IP].
Please draft a privacy policy for McLaughlin - Dicki's website that covers user data protection, including Cristina93, FIrrydsgrevq, and 54.40.51.

In [68]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IPV6\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [70]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IPV4\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [71]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[MAC\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBDESCRIPTOR\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Create a guide for [FULLNAME] on conducting effective performance reviews for their [JOBDESCRIPTOR] role.
Create a guide for Randal Labadie on conducting effective performance reviews for their Forward role.


Create a list of compliance laws related to the [JOBDESCRIPTOR] industry for [FULLNAME] to review.
Create a list of compliance laws related to the National industry for Emily Nienow to review.


Hey, can you suggest a few books or resources for [FULLNAME] to improve their [JOBDESCRIPTOR] skills?
Hey, can you suggest a few books or resources for Sheila Herman PhD to improve their Investor skills?


Can you prepare a risk management report on the [JOBDESCRIPTOR] industry for [FULLNAME]?
Can you prepare a risk management report on the Legacy industry for Mrs. Stella Kassulke?


Prepare a list of effective team-building activities for [FULLNAME] to implement in their [JOBDESCRIPTOR] role.
Prepare a list of effective team-building activities for Dr. Nina Gutmann V to implement in the

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBTYPE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Please provide a summary of the legal responsibilities of [FULLNAME] as a director of a [JOBTYPE] company in the [JOBAREA] sector.
Please provide a summary of the legal responsibilities of Bert Feest-Sporer as a director of a Designer company in the Response sector.


Develop a comprehensive risk assessment questionnaire for [FULLNAME] to collect data on their [JOBTYPE] company's vulnerabilities.
Develop a comprehensive risk assessment questionnaire for Crystal Anderson to collect data on their Technician company's vulnerabilities.


9. Design a corporate social responsibility (CSR) policy for [FULLNAME]'s [JOBTYPE] company.
9. Design a corporate social responsibility (CSR) policy for Sheldon Weimann's Assistant company.


I need a breakdown of the civil rights issues faced by [JOBTYPE] workers in [STATE]. Please send it to [EMAIL].
I need a breakdown of the civil rights issues faced by Officer workers in Kansas. Please send it to Sarina.Hoppe@yahoo.com.


3. Write a corporate governa

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBTITLE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Create a list of the top 10 healthcare law cases in [STATE] that [FULLNAME], a [JOBTITLE], should be aware of.
Create a list of the top 10 healthcare law cases in Arkansas that Spencer Ward, a Future Markets Facilitator, should be aware of.


Write an article about the role of [JOBTITLE] in criminal law for [FULLNAME]'s law blog.
Write an article about the role of International Tactics Administrator in criminal law for Henrietta Kilback's law blog.


14. How [FULLNAME] can use psychometric tests to identify potential [JOBTITLE] candidates.
14. How Gordon Leffler Sr. can use psychometric tests to identify potential Direct Identity Facilitator candidates.


8. Write a contract for [FULLNAME] to hire [FULLNAME] as a [JOBTITLE] in their company.
8. Write a contract for Geraldine Runolfsdottir to hire Lamar Krajcik as a Product Group Director in their company.


In the video conference, can you explain the role of [JOBTITLE] in operations management at [FULLNAME]'s company?
In the video co

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ORDINALDIRECTION\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 7):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Would you mind assisting me with a research paper on the legal ethics of accessing [ORDINALDIRECTION] and [NEARBYGPSCOORDINATE] for navigation purposes?
Would you mind assisting me with a research paper on the legal ethics of accessing Southwest and [76.1191,-133.7187] for navigation purposes?


15. Develop a YouTube marketing strategy for [NAME] targeting viewers in [ORDINALDIRECTION].
15. Develop a YouTube marketing strategy for Christiansen, Mann and Orn targeting viewers in Southeast.


16. Write a blog post on the benefits of offering various shipping options, such as [ORDINALDIRECTION] and [ORDINALDIRECTION] shipping.
16. Write a blog post on the benefits of offering various shipping options, such as Northeast and Northeast shipping.


2. Develop an email marketing strategy for [NAME] with a focus on [ORDINALDIRECTION] region.
2. Develop an email marketing strategy for Kuhn Group with a focus on Southeast region.


Draft a risk management plan for [NAME] to address potential dis

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[NUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Please provide a list of [NUMBER] recommended books on child psychology for [NAME] to read.
Please provide a list of 1-422-747-6636 recommended books on child psychology for Luettgen - Feil to read.


Can you provide a list of items in the inventory with expiration dates within the next [NUMBER] days?
Can you provide a list of items in the inventory with expiration dates within the next 656-512-2545 days?


Create an infographic for [NAME] illustrating the major milestones in civil rights law over the past [NUMBER] years.
Create an infographic for Nitzsche, Schuster and Moen illustrating the major milestones in civil rights law over the past 645.486.2660 x06882 years.


Draft an email invitation for [FULLNAME] to attend a psycho-oncology conference in [CITY] on [NUMBER].
Draft an email invitation for Ebony Wilderman III to attend a psycho-oncology conference in Eduardoview on 1-980-891-4736 x429.


I'd like to create a more efficient organizational design for our office. We have [NUMB

In [75]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[MASKEDNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0,20):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Please provide a list of [MASKEDNUMBER] recommended books on child psychology for [NAME] to read.
Please provide a list of 1-422-747-6636 recommended books on child psychology for Luettgen - Feil to read.


Help me create an expense report for my credit card with number ending in [MASKEDNUMBER], please.
Help me create an expense report for my credit card with number ending in 1789939217688260, please.


Can you provide a list of items in the inventory with expiration dates within the next [MASKEDNUMBER] days?
Can you provide a list of items in the inventory with expiration dates within the next 656-512-2545 days?


Please write an article on the impact of insurance law changes for policyholders with account numbers like [MASKEDNUMBER].
Please write an article on the impact of insurance law changes for policyholders with account numbers like 20933791.


Write a report on the impact of rehabilitation psychology in the recovery process of patients with account numbers [MASKEDNUMBER], [MA

In [73]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[AMOUNT\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [72]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ACCOUNTNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

No rows found with the token 


In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYSYMBOL\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Assess the distribution agreement for [FULLNAME] and provide recommendations for changes, including the use of [CURRENCYNAME] and [CURRENCYSYMBOL].
Assess the distribution agreement for Jessica Parker and provide recommendations for changes, including the use of Turkish Lira and $.


Can you help me create a risk management framework for a company with an annual revenue of [AMOUNT][CURRENCYSYMBOL]?
Can you help me create a risk management framework for a company with an annual revenue of 895.50$?


Could you help me draft a code of conduct for legal professionals handling cases involving clients' [CURRENCY] and [CURRENCYSYMBOL] transactions?
Could you help me draft a code of conduct for legal professionals handling cases involving clients' Ethiopian Birr and Ft transactions?


Hey, can you help me create a risk appetite statement for a company with a net worth of [AMOUNT][CURRENCYSYMBOL]?
Hey, can you help me create a risk appetite statement for a company with a net worth of 13.63$?



In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Assess the distribution agreement for [FULLNAME] and provide recommendations for changes, including the use of [CURRENCYNAME] and [CURRENCYSYMBOL].
Assess the distribution agreement for Jessica Parker and provide recommendations for changes, including the use of Turkish Lira and $.


Evaluate the merger agreement between [FULLNAME] and [FULLNAME] and provide feedback on any potential issues, including liabilities and [CURRENCYNAME] payment terms.
Evaluate the merger agreement between Victoria Larkin and Patrick Kirlin and provide feedback on any potential issues, including liabilities and Boliviano boliviano payment terms.


Help [NAME] create a resume tailored for their work visa application in [CURRENCYNAME].
Help Beier, Moen and Ondricka create a resume tailored for their work visa application in Kenyan Shilling.


Assist [NAME] in writing a cover letter for their skilled worker visa application in [CURRENCYNAME].
Assist Maggio, Oberbrunner and McDermott in writing a cover letter f

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCY\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


During our video conference, please provide a detailed overview of [FULLNAME]'s company's balance sheet and the impact of [CURRENCY] exchange rates.
During our video conference, please provide a detailed overview of Lydia Grant PhD's company's balance sheet and the impact of Kina exchange rates.


4. Write a financial risk assessment for [FULLNAME]'s investment portfolio, including assets in [CURRENCY] and [CURRENCY].
4. Write a financial risk assessment for Felicia Runte's investment portfolio, including assets in Zambian Kwacha and Saudi Riyal.


13. Organize a media event with a budget of [AMOUNT] [CURRENCY] to showcase the company's latest product.
13. Organize a media event with a budget of 933.83 Trinidad and Tobago Dollar to showcase the company's latest product.


Can you analyze [NAME]'s current expenses and suggest ways to cut costs by [CURRENCY][AMOUNT]?
Can you analyze Walter, Feil and Predovic's current expenses and suggest ways to cut costs by Pound Sterling891.75?


I n

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYCODE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Assess the vendor agreement for [FULLNAME] and provide feedback on any potential risks, including liability and [CURRENCYCODE] payment terms.
Assess the vendor agreement for Gene Wilderman and provide feedback on any potential risks, including liability and ALL payment terms.


Draft a press release for [NAME] announcing the launch of their new tax consultancy firm, focusing on [CURRENCYCODE] transactions.
Draft a press release for Grant, Crist and Gerlach announcing the launch of their new tax consultancy firm, focusing on CAD transactions.


Are there any specific banking laws for transactions involving [CURRENCYCODE]?
Are there any specific banking laws for transactions involving CLP?


Please analyze the ethical implications of using [CURRENCY] and [CURRENCYCODE] in international legal disputes.
Please analyze the ethical implications of using Singapore Dollar and OMR in international legal disputes.


16. Identify potential risks to sales projections, such as fluctuations in curr

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[BITCOINADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)


Draft an email to [NAME] at [EMAIL] discussing the legal implications of using cryptocurrencies like [BITCOINADDRESS] and [ETHEREUMADDRESS] in international trade.
Draft an email to Schmitt Inc at Everardo_Bahringer94@hotmail.com discussing the legal implications of using cryptocurrencies like 3YB5DGTab7DGXGztwPzEHx8QEfUnHffd and 0xaa93f0df8d82b08c8caaeb2d2234efbab4ffc96d in international trade.


How can I protect my [BITCOINADDRESS] and [ETHEREUMADDRESS] from hackers and scammers?
How can I protect my 1phPEkCpeFhLtdeckBM2sVSBFMcLK8jzY3hv and 0x62abe971fb5e968dcf0ceeda7fb61880fc0750cf from hackers and scammers?


What are the tax implications for a small business owner who accepts payments in cryptocurrencies like Bitcoin, with addresses like [BITCOINADDRESS]?
What are the tax implications for a small business owner who accepts payments in cryptocurrencies like Bitcoin, with addresses like 3LvEmePsFj57CKx2k4jTExGHxvVW1hfgwNc?


Draft a legal opinion letter for [NAME] regarding the se

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ETHEREUMADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


Draft an email to [NAME] at [EMAIL] discussing the legal implications of using cryptocurrencies like [BITCOINADDRESS] and [ETHEREUMADDRESS] in international trade.
Draft an email to Schmitt Inc at Everardo_Bahringer94@hotmail.com discussing the legal implications of using cryptocurrencies like 3YB5DGTab7DGXGztwPzEHx8QEfUnHffd and 0xaa93f0df8d82b08c8caaeb2d2234efbab4ffc96d in international trade.


How can I protect my [BITCOINADDRESS] and [ETHEREUMADDRESS] from hackers and scammers?
How can I protect my 1phPEkCpeFhLtdeckBM2sVSBFMcLK8jzY3hv and 0x62abe971fb5e968dcf0ceeda7fb61880fc0750cf from hackers and scammers?


Could you please write a short story about a lawyer dealing with a case involving cryptocurrency, specifically [BITCOINADDRESS] and [ETHEREUMADDRESS]?
Could you please write a short story about a lawyer dealing with a case involving cryptocurrency, specifically 1DgA6jHyUCJUtLoca9AzmwRYBJFRE9ifCeWa and 0x3ac6caec3b28a29cdca8ff97efd4cf123c5b95f5?


Can you help me understand t

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[LITECOINADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


Can you explain the legal aspects of using a [LITECOINADDRESS] for illegal activities?
Can you explain the legal aspects of using a MJtXjuvDLmhaRhqyp8w4ji86suP6qqCq for illegal activities?


Hey, can you help me understand the legal implications of hacking into a Litecoin wallet, like [LITECOINADDRESS]?
Hey, can you help me understand the legal implications of hacking into a Litecoin wallet, like MMJ3Qj9LEaRsg2849FzXeWGN5G2h?


Write a case study on a cyber law violation involving unauthorized access to an individual's [BITCOINADDRESS] and [LITECOINADDRESS].
Write a case study on a cyber law violation involving unauthorized access to an individual's 3d4CV2AGxpur1vzjdaWxRSNn15F and LvdEBfW8t5M5QYPzDvx52fKo3j.


Let's create a business continuity plan for [NAME] to protect their assets like [LITECOINADDRESS] from potential risks and disruptions in the cryptocurrency market.
Let's create a business continuity plan for Kris and Sons to protect their assets like L1AJfnAy9hyHRAnRepr2akDQFmW

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CREDITCARDCVV\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


Prepare a presentation on the legal aspects of protecting sensitive customer information like [CREDITCARDNUMBER] and [CREDITCARDCVV].
Prepare a presentation on the legal aspects of protecting sensitive customer information like 2463651259605690 and 367.


Design a risk mitigation plan for [NAME] to address potential credit card fraud, focusing on [CREDITCARDNUMBER] and [CREDITCARDCVV].
Design a risk mitigation plan for Weimann - Schinner to address potential credit card fraud, focusing on 0578471333420469 and 997.


What are the laws regarding phishing attacks that target [CREDITCARDNUMBER] and [CREDITCARDCVV]?
What are the laws regarding phishing attacks that target 8866227781842689 and 956?


Draft an email to [NAME] at [EMAIL] with tips on how to safeguard their credit card information, including [CREDITCARDNUMBER] and [CREDITCARDCVV].
Draft an email to McLaughlin - Pfeffer at Vada29@gmail.com with tips on how to safeguard their credit card information, including 7216485385871965 a

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[PIN\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


I'd like to know the legal implications of accessing someone's bank account with their [ACCOUNTNUMBER] and [PIN] without permission.
I'd like to know the legal implications of accessing someone's bank account with their 39014393 and 0687 without permission.


I could use your help in writing a report on the ethical considerations of using [ACCOUNTNAME] and [PIN] in legal investigations.
I could use your help in writing a report on the ethical considerations of using Credit Card Account and 5467 in legal investigations.


Let's create a list of best practices for safeguarding sensitive information like [ACCOUNTNUMBER] and [PIN].
Let's create a list of best practices for safeguarding sensitive information like 29307602 and 3198.


What are the best practices for maintaining user privacy when handling [PASSWORD] and [PIN] data?
What are the best practices for maintaining user privacy when handling 8_r97wUAWnDq and 5995 data?


What are the main legal considerations for banks when dealing

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CREDITCARDISSUER\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


Hey, I need some guidance on securing a business loan for our company. Can you provide information on the required documents and credit history for [CREDITCARDISSUER]?
Hey, I need some guidance on securing a business loan for our company. Can you provide information on the required documents and credit history for discover?


Please create a step-by-step guide for [NAME] on setting up a secure payment gateway using [CREDITCARDISSUER] for their eCommerce store.
Please create a step-by-step guide for Ryan - Roob on setting up a secure payment gateway using laser for their eCommerce store.


Can you find an eating disorder therapy center near [BUILDINGNUMBER] [STREET] that accepts [CREDITCARDISSUER] credit cards?
Can you find an eating disorder therapy center near 1459 Effertz Manor that accepts diners_club credit cards?


Prepare a presentation on the role of cyber law in securing [CREDITCARDNUMBER] and [CREDITCARDISSUER] information during online transactions.
Prepare a presentation on

In [ ]:
# 
# def safe_detect(text):
# 	try:
# 		return detect(text)
# 	except Exception:
# 		return None
# 
# # Detect language for each text in the 'Filled Template' column
# df_full['language'] = df_full['Filled Template'].apply(safe_detect)
# 
# # Display a sample of the results
# print(df_full[['Filled Template', 'language']].head())
# 
# # Save the updated DataFrame to a pickle file.
# with open("df_full_language.pkl", "wb") as f:
# 	pickle.dump(df_full, f)
# 

In [ ]:


# Load the DataFrame from the pickle file.
with open("df_full_language.pkl", "rb") as f:
	df_full = pickle.load(f)

# Display a sample of the loaded results.
print(df_full[['Filled Template', 'language']].head())

                                     Filled Template language
0  In our video conference, discuss the role of e...       en
1  Could you draft a letter for Dietrich, Schulis...       en
2  Discuss the options for Jeffery Pfeffer who wa...       en
3  13. Write a press release announcing Gayle Wat...       en
4  9. Develop an inventory management plan for Ev...       en


In [ ]:
unique_languages = df_full['language'].unique()
print("Languages present in 'language' column:", unique_languages)

Languages present in 'language' column: ['en' 'fr' 'nl' 'es' 'ca']


In [ ]:
lang_map = {
	'en': 'English',
	'fr': 'French',
	'ca': 'Catalan',
	'nl': 'Dutch',
	'es': 'Spanish'
}

for lang in unique_languages:
	language_name = lang_map.get(lang, lang)
	examples = df_full[df_full['language'] == lang]["Filled Template"].head(5)
	print(f"Examples for {language_name}:")
	for text in examples:
		print("-", text)
	print("\n" + "-"*40 + "\n")

Examples for English:
- In our video conference, discuss the role of evidence in the arbitration process involving Dr. Marvin Rolfson and Julius Daugherty.
- Could you draft a letter for Dietrich, Schulist and Romaguera to send to their insurance company regarding a claim dispute under Washington law?
- Discuss the options for Jeffery Pfeffer who wants to obtain a green card through employment in Port Ron.
- 13. Write a press release announcing Gayle Waters's new mindfulness-based therapy practice in Oceanside.
- 9. Develop an inventory management plan for Evan Anderson that ensures stock levels are maintained across multiple Mayaguez locations.

----------------------------------------

Examples for French:
- Develop a comprehensive questionnaire for Elaine Schuppe to collect data on children's social skills development.
- 11. Create a stress management assessment questionnaire for Dr. Elbert Bauch to fill out.
- I'd appreciate it if you could prepare a financial risk assessment for A

In [102]:
import re

emoji_pattern = re.compile(
	"["
	"\U0001F600-\U0001F64F"  # emoticons
	"\U0001F300-\U0001F5FF"  # symbols & pictographs
	"\U0001F680-\U0001F6FF"  # transport & map symbols
	"\U0001F1E0-\U0001F1FF"  # flags
	"]+", flags=re.UNICODE)

# Check for emojis in the 'Filled Template' column of df_full
df_with_emoji = df_full[df_full['Filled Template'].apply(lambda text: bool(emoji_pattern.search(text)))]
if df_with_emoji.empty:
	print("No emojis found in the dataset.")
else:
	print("Emojis found in the dataset. Examples:")
	print(df_with_emoji['Filled Template'].head())

No emojis found in the dataset.
